# Walmart Store Sales Forecasting

Importa as bibliotecas e lista os arquivos de input da competição.

In [ ]:
import os, numpy as np, pandas as pd
import lightgbm as lgb

INPUT = '/kaggle/input/walmart-recruiting-store-sales-forecasting'
print(os.listdir(INPUT))

Carrega os CSVs (aceita tanto `.csv` quanto `.csv.zip`).

In [ ]:
def read(name):
    for path in [f'{INPUT}/{name}.csv', f'{INPUT}/{name}.csv.zip']:
        if os.path.exists(path):
            return pd.read_csv(path)
    raise FileNotFoundError(name)

train = read('train')
test = read('test')
features = read('features')
stores = read('stores')

Junta `stores` (tipo/tamanho da loja) e `features` (temperatura, preço do combustível, CPI, etc.) às bases de treino e teste.

In [ ]:
train = train.merge(stores, on='Store', how='left')
test = test.merge(stores, on='Store', how='left')

features = features.drop(columns=['IsHoliday'])
train = train.merge(features, on=['Store','Date'], how='left')
test = test.merge(features, on=['Store','Date'], how='left')

Cria colunas de data (ano, mês, semana, dia), converte `Type` para número, `IsHoliday` para inteiro e preenche valores faltantes.

In [ ]:
def add_date_feats(df):
    df['Date'] = pd.to_datetime(df['Date'])
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
    df['Day'] = df['Date'].dt.day
    return df

train = add_date_feats(train)
test = add_date_feats(test)

for df in [train, test]:
    df['Type'] = df['Type'].map({'A':0,'B':1,'C':2})
    df['IsHoliday'] = df['IsHoliday'].astype(int)
    for md in ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']:
        df[md] = df[md].fillna(0)
    df['CPI'] = df['CPI'].fillna(df['CPI'].median())
    df['Unemployment'] = df['Unemployment'].fillna(df['Unemployment'].median())

Define as features usadas pelo modelo e separa `X` (entradas), `y` (alvo) e `X_test`.

In [ ]:
features_cols = ['Store','Dept','Type','Size','Year','Month','Week','Day',
                  'IsHoliday','Temperature','Fuel_Price','CPI','Unemployment',
                  'MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']

X = train[features_cols]
y = train['Weekly_Sales']
X_test = test[features_cols]

Implementa a métrica WMAE (erro absoluto médio ponderado, com peso 5x em semanas de feriado — métrica oficial da competição). Treina um LightGBM em parte do histórico e valida nas últimas 8 semanas para checar a métrica.

In [ ]:
def wmae(y_true, y_pred, is_holiday):
    w = np.where(is_holiday==1, 5, 1)
    return np.sum(w*np.abs(y_true-y_pred))/np.sum(w)

cut = train['Date'].max() - pd.Timedelta(weeks=8)
tr_idx = train['Date'] <= cut
va_idx = train['Date'] > cut

model = lgb.LGBMRegressor(n_estimators=1200, learning_rate=0.03, num_leaves=128,
                           subsample=0.8, colsample_bytree=0.8, random_state=42)
model.fit(X[tr_idx], y[tr_idx])
val_pred = model.predict(X[va_idx])
print('WMAE val:', wmae(y[va_idx].values, val_pred, X.loc[va_idx,'IsHoliday'].values))

Re-treina o modelo usando todo o histórico disponível e gera as previsões para o conjunto de teste (sem valores negativos).

In [ ]:
model_full = lgb.LGBMRegressor(n_estimators=1200, learning_rate=0.03, num_leaves=128,
                                subsample=0.8, colsample_bytree=0.8, random_state=42)
model_full.fit(X, y)
preds = model_full.predict(X_test)
preds = np.maximum(preds, 0)

Monta o `submission.csv` no formato exigido (`Id` = Store_Dept_Data e `Weekly_Sales` previsto).

In [ ]:
sub = pd.DataFrame({
    'Id': test['Store'].astype(str) + '_' + test['Dept'].astype(str) + '_' + test['Date'].dt.strftime('%Y-%m-%d'),
    'Weekly_Sales': preds
})
sub.to_csv('submission.csv', index=False)
sub.head()